                # COPRO

                Propose instruction variants, evaluate them, and iteratively refine the best candidates.

                **Use it when:** The instruction is likely the bottleneck and you want a direct, interpretable prompt search without demo search.

                **What compilation changes:** Instruction text only; the selected program has no demonstrations in this benchmark.

                Important configuration in this benchmark:

                - full-profile breadth 4 and depth 2
- GPT-5.6 Sol proposes; Luna executes
- validation uses the exact-match metric

                Every notebook loads the same 74-row dataset and frozen, pair-grouped
                train/validation/test membership before it can compile anything.
                The test partition is deliberately baseline-adversarial, so these scores teach
                optimizer tradeoffs; they are not a general-purpose AI-detector leaderboard.

In [1]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "chapter06").is_dir() else cwd.parent
if not (REPO_ROOT / "chapter06" / "results" / "benchmark_summary.json").exists():
    raise RuntimeError("Run this notebook from the repository or chapter06 directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from chapter06.notebook_support import artifact_paths, learned_program_preview, verify_prompt_artifact
from chapter06.optimizer_runtime import (
    format_result,
    load_frozen_examples,
    published_result,
    run_optimizer,
    split_summary,
)

OPTIMIZER = 'copro'
splits = load_frozen_examples()
RUN_LIVE = os.getenv("CHAPTER06_RUN_LIVE", "0") == "1"
print(f"optimizer={OPTIMIZER!r}; live={RUN_LIVE}")
print(split_summary(splits))

optimizer='copro'; live=False
train=36 (human=18, AI=18); validation=18 (human=9, AI=9); test=20 (human=10, AI=10)


                ## Compile shape

                This is the essential DSPy call used by the shared executable runner:

                ```python
                optimizer = dspy.COPRO(
    prompt_model=reflection_lm, metric=exact_match,
    breadth=profile.copro_breadth, depth=profile.copro_depth, track_stats=True,
)
optimized_detector = optimizer.compile(detector, trainset=trainset)
                ```

                `compile` returns a program. The shared runner then evaluates that program on the
                untouched 20-example test split. The baseline has its own notebook; all other
                notebooks report the optimized program's final test accuracy directly.

In [2]:
if RUN_LIVE:
    live_run = run_optimizer(
        OPTIMIZER,
        splits=splits,
    )
    result = live_run.summary()
else:
    result = published_result(OPTIMIZER)

print(format_result(result))
print()
print(artifact_paths(OPTIMIZER))

optimizer: copro
task model: openai/gpt-5.6-luna
final test accuracy: 50.0% (10/20)
optimization time: 755.0s

Published artifacts:
- canonical program snapshot: chapter06/optimized_programs/final/copro.json
- canonical prompt: chapter06/results/final_prompts/copro.json
- chapter comparison: chapter06/CHAPTER_RESULTS.md


## Read the result

Read the learned instruction before the score. COPRO's artifact is especially auditable because its gain must come from wording rather than hidden examples.

The saved output above uses the checked-in result so opening or running the notebook
is cheap. Set `CHAPTER06_RUN_LIVE=1` before launching Jupyter to execute the real
optimizer; prompt optimizers require an OpenAI key, while weight optimizers also
require the local PyTorch/Transformers stack. The next cell previews the published
program artifact.

In [3]:
print(learned_program_preview(OPTIMIZER))
print()
print("Frozen program/prompt consistency:", verify_prompt_artifact(OPTIMIZER))

Predictor: detect.predict
Learned instruction (509 characters):
Analyze the supplied passage for signals associated with AI-generated versus human-written text, including repetitiveness, formulaic structure, generic phrasing, unnatural consistency, unsupported specificity, stylistic variation, and coherent personal context. Weigh the evidence on both sides and make the best binary classification, while recognizing that authorship cannot be determined with certainty from prose alone. Return only `AI-generated` or `Human-written`; do not add explanations or qualifiers.

Demonstrations: 0

Frozen program/prompt consistency: {'checked': True, 'predictors': 1, 'prompt_state_equal': True, 'mismatches': []}


## Apply the pattern

Adapt the compile shape above to your own DSPy program, metric, and frozen
train/validation split. Keep the test set untouched until the optimizer returns,
then report final accuracy as `correct / test examples` so every optimizer is easy
to compare. Use the separate baseline notebook when you also need uplift.

The complete Chapter 6 rerun is summarized in `CHAPTER_RESULTS.md`. Raw provider
transcripts and temporary training outputs are intentionally excluded from the
student download.